In [ ]:
import torch 
# import torchviz 
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
import os 
import sys 
sys.path.append('../../python')
import glob
import logging
from torch.nn.functional import interpolate
from ddms.surrogate import Trainer, Tracker
from ddms.surrogate.model import *
from ddms.surrogate.dataset import *
from ddms.surrogate.loss import *
from ddms.surrogate.mechanics import *
from ddms.surrogate.tensor import *
from ddms.surrogate.processing import *
from ddms.surrogate.visualize import *

In [ ]:
%matplotlib inline

### global variables

In [ ]:
DATA_ROOT = r''
DATA_NAME = 'morient_20_0_tensionx'
result_path = os.path.join(DATA_ROOT, 'result', DATA_NAME)
hdf5_path = glob.glob(f'{result_path}/*.hdf5')[0]

res     = Result(hdf5_path, ['F_e', 'F_p', 'P', 'dPdF'])
Fe 		= res.get('F_e')
Fp 		= res.get('F_p')
P 		= res.get('P')													# (seq_len, num_grid, 3, 3)
dPdF 	= res.get('dPdF')												# (seq_len, num_grid, 81)

F		= FeFp_to_F(Fe, Fp)												# (seq_len, num_grid, 3, 3)
CS 		= P_to_CS(P, F)													# (seq_len, num_grid, 3, 3)
dCSdE 	= dPdF_to_dCSdE(dPdF, P, F)				                        # (seq_len, num_grid, 6, 6)
result 	= FCS_to_dev5(F, CS, dim=0)
result.keys()

In [ ]:
def predict(dHSdev5=None, CShyd6=None, return_aux=False, **kwargs):
	args = Args(
		job_type		= 'predict',
		exp_name 		= kwargs.get('exp_name', 'LMSC'),
		run_id 			= kwargs.get('run_id', 'olrdx1ao'),
		data_path 		= kwargs.get('data_path', ''),
		save_root		= kwargs.get('save_root', ''),
		specify_data 	= kwargs.get('specify_data', ['tensionx']),
		backend			= kwargs.get('backend', 'wandb'),
		device 			= kwargs.get('device', 'cpu'),
		level			= logging.ERROR,
	)
	return Trainer(args).predict(LMSC, MemoryDataset_LSTM, {}, dHSdev5, CShyd6, return_aux)

In [ ]:
def get_trainer(**kwargs):
    args = Args(
		job_type		= 'predict',
		exp_name 		= kwargs.get('exp_name', 'LMSC'),
        run_name		= kwargs.get('run_name', 'NJA800'),
		run_id 			= kwargs.get('run_id', 'olrdx1ao'),
		data_path 		= kwargs.get('data_path', ''),
		specify_data 	= kwargs.get('specify_data', ['tensionx']),
		device 			= kwargs.get('device', 'cpu'),
		level			= logging.ERROR,
	)
    trainer = Trainer(args)
    trainer.set_dataset(MemoryDataset_LSTM)
    trainer.set_model(LMSC, kwargs.get('hyperparams', {}), kwargs.get('broken', False))
    trainer.set_loss()
    trainer.set_optimizer()
    return trainer

### fig4_datasest_input_output

In [ ]:
# strain stress
eps6 = sym33_to_m6(result['eps'].mean(dim=1), MAP=MAP.m33toEXP6)       # (s, 6)
CS6 = sym33_to_m6(result['sigma'].mean(dim=1), MAP=MAP.m33toEXP6)      # (s, 6)

# # input / output 
deps_dev5 = result['dHSdev5'].mean(dim=1)                               # (s-1, 5)
CSdev5 = result['CSdev5'].mean(dim=1)                                   # (s-1, 5)

In [ ]:
vals = [eps6, CS6/1e6, deps_dev5, CSdev5/1e6]
ylabels = ['total strain[-]', 'stress[MPa]', 'inputs[-]', 'outputs[MPa]']
fig, axes = plt.subplots(2, 2, figsize=(8,6), tight_layout=True)

for i, (val, ylabel, ax) in enumerate(zip(vals, ylabels, axes.flatten())):
    ax.plot(np.arange(len(val)), val)
    ax.set_xlabel('increment[-]')
    ax.set_ylabel(ylabel)
plt.show()

In [ ]:
x_norm = torch.linalg.norm(deps_dev5, dim=-1, keepdim=True).clamp(1e-15, 100)

In [ ]:
plt.figure(figsize=(10,5))
plt.hist(x_norm.flatten())
plt.show()

In [ ]:
data = torch.load('')[0]
x_norm = torch.linalg.norm(data.dHSdev5, dim=-1, keepdim=True).clamp(1e-15, 100)

In [ ]:
plt.figure(figsize=(10,5))
plt.hist(x_norm.flatten(), bins=200, range=[0,0.01], density=True)
plt.show()

### fig4_tune_hyperparameters
- training loss increase with layer increase, probably caused by
    - gradient vanishing
    - tanh activation, saturate
    - ref: Understanding the difficulty of training deep feedforward neural networks

In [ ]:
path = r''
name = 'LMSC_tune.xlsx'
df_CS = pd.read_excel(os.path.join(path, name), sheet_name='architecture', index_col=0, header=0).sort_values(by='hid_ch')
df_CS.head()

#### parameters vs. MAE

In [ ]:
stt_chs = [8,16,32,64,128]
markers = ['o', 's', 'v', '^', '*']

fig, axes = plt.subplots(1, 3, figsize=(10, 3), tight_layout=True)
for stt_ch, marker in zip(stt_chs, markers):
    mask = df_CS['stt_ch']==stt_ch
    axes[0].scatter(df_CS[mask]['parameters'], df_CS[mask]['train/loss'], marker=marker)
    axes[1].scatter(df_CS[mask]['parameters'], df_CS[mask]['val/loss'], marker=marker)
    axes[2].scatter(df_CS[mask]['parameters'], df_CS[mask]['test/loss'], marker=marker)

axes[0].set_ylim([0.01,0.15])
axes[1].set_ylim([0.05,0.11])
axes[2].set_ylim([0.035,0.075])

for i, ax in enumerate(axes):
    ax.set_xlabel('parameters')
    ax.set_ylabel('mean absolute error')
    ax.set_xscale('log')
    ax.set_yscale('log')

vtxt_kwargs = {'rotation': 'horizontal', 'va': 'center', 'weight': 'bold', 'fontsize': 12}
fig.text(0.175, 1.01, 'train', **vtxt_kwargs)
fig.text(0.48, 1.01, 'validataion', **vtxt_kwargs)
fig.text(0.85, 1.01, 'test', **vtxt_kwargs)
fig.legend(stt_chs, title='state variables', loc='lower center', bbox_to_anchor=(0.5,-0.15), ncol=len(stt_chs), alignment='center', frameon=True)
plt.show()

#### parameters vs. correlation

In [ ]:
stt_chs = [8,16,32,64,128]
markers = ['o', 's', 'v', '^', '*']

fig, axes = plt.subplots(1, 3, figsize=(10, 3), tight_layout=True)
for stt_ch, marker in zip(stt_chs, markers):
    mask = df_CS['stt_ch']==stt_ch
    axes[0].scatter(df_CS[mask]['parameters'], df_CS[mask]['val/corr44'], marker=marker)
    axes[1].scatter(df_CS[mask]['parameters'], df_CS[mask]['val/corr55'], marker=marker)
    axes[2].scatter(df_CS[mask]['parameters'], df_CS[mask]['val/corr66'], marker=marker)

# axes[0].set_ylim([0.98,1])
# axes[1].set_ylim([0.98,1])
# axes[2].set_ylim([0.98,1])

for i, ax in enumerate(axes):
    ax.set_xlabel('parameters')
    ax.set_ylabel('mean absolute error')
    ax.set_xscale('log')
    # ax.set_yscale('log')

vtxt_kwargs = {'rotation': 'horizontal', 'va': 'center', 'weight': 'bold', 'fontsize': 12}
fig.text(0.175, 1.01, 'train', **vtxt_kwargs)
fig.text(0.48, 1.01, 'validataion', **vtxt_kwargs)
fig.text(0.85, 1.01, 'test', **vtxt_kwargs)
fig.legend(stt_chs, title='state variables', loc='lower center', bbox_to_anchor=(0.5,-0.15), ncol=len(stt_chs), alignment='center', frameon=True)
plt.show()

#### hyperparameters vs. MAE (legacy)

In [ ]:
hid_chs = [8,16,32,64,128]
stt_chs = [8,16,32,64,128]

fig, axes = plt.subplots(3, 3, figsize=(10,8), tight_layout=True)
for stt_ch in stt_chs:
    mask = np.logical_and(df_CS['stt_ch']==stt_ch, df_CS['n_layer']==1)
    axes[0,0].plot(df_CS[mask]['hid_ch'], (df_CS[mask]['train/loss']), '--o')
    axes[1,0].plot(df_CS[mask]['hid_ch'], (df_CS[mask]['val/loss_legacy']), '--o')
    axes[2,0].plot(df_CS[mask]['hid_ch'], (df_CS[mask]['test/loss_legacy']), '--o')

    mask = np.logical_and(df_CS['stt_ch']==stt_ch, df_CS['n_layer']==3)
    axes[0,1].plot(df_CS[mask]['hid_ch'], (df_CS[mask]['train/loss']), '--o')
    axes[1,1].plot(df_CS[mask]['hid_ch'], (df_CS[mask]['val/loss_legacy']), '--o')
    axes[2,1].plot(df_CS[mask]['hid_ch'], (df_CS[mask]['test/loss_legacy']), '--o')

    mask = np.logical_and(df_CS['stt_ch']==stt_ch, df_CS['n_layer']==5)
    axes[0,2].plot(df_CS[mask]['hid_ch'], (df_CS[mask]['train/loss']), '--o')
    axes[1,2].plot(df_CS[mask]['hid_ch'], (df_CS[mask]['val/loss_legacy']), '--o')
    axes[2,2].plot(df_CS[mask]['hid_ch'], (df_CS[mask]['test/loss_legacy']), '--o')

for i, ax in enumerate(axes.flatten()):
    if i<3:
        ax.set_ylim([0.01,0.4])
    elif i>=3 and i<6:
        ax.set_ylim([0.065,0.09])
    else:
        ax.set_ylim([0.035,0.07])
        ...

    ax.set_yscale('log')
    ax.xaxis.set_ticks(hid_chs)
    ax.yaxis.set_tick_params(pad=20)
    ax.set_xlabel('hidden features')
    ax.set_ylabel('mean absolute error')


htxt_kwargs = {'rotation': 'horizontal', 'ha': 'center', 'weight': 'bold', 'fontsize': 12}
fig.text(0.21, 0.99, r'$\mathbf{d=1}$', **htxt_kwargs)
fig.text(0.54, 0.99, r'$\mathbf{d=3}$', **htxt_kwargs)
fig.text(0.87, 0.99, r'$\mathbf{d=5}$', **htxt_kwargs)

vtxt_kwargs = {'rotation': 'vertical', 'va': 'center', 'weight': 'bold', 'fontsize': 12}
fig.text(-0.01, 0.85, 'train', **vtxt_kwargs)
fig.text(-0.01, 0.525, 'validataion', **vtxt_kwargs)
fig.text(-0.01, 0.2, 'test', **vtxt_kwargs)
fig.legend(stt_chs, title='state variables', loc='lower center', bbox_to_anchor=(0.5,-0.05), ncol=len(stt_chs), alignment='center', frameon=True)
plt.show()

#### hyperparameters vs. MAE

In [ ]:
hid_chs = [8,16,32,64,128]
stt_chs = [8,16,32,64,128]

fig, axes = plt.subplots(3, 3, figsize=(10,8), tight_layout=True)
for stt_ch in stt_chs:
    mask = np.logical_and(df_CS['stt_ch']==stt_ch, df_CS['n_layer']==1)
    axes[0,0].plot(df_CS[mask]['hid_ch'], (df_CS[mask]['val/loss']), '--o')
    axes[1,0].plot(df_CS[mask]['hid_ch'], (df_CS[mask]['valsmooth/loss']), '--o')
    axes[2,0].plot(df_CS[mask]['hid_ch'], (df_CS[mask]['valstiff/loss']), '--o')

    mask = np.logical_and(df_CS['stt_ch']==stt_ch, df_CS['n_layer']==3)
    axes[0,1].plot(df_CS[mask]['hid_ch'], (df_CS[mask]['val/loss']), '--o')
    axes[1,1].plot(df_CS[mask]['hid_ch'], (df_CS[mask]['valsmooth/loss']), '--o')
    axes[2,1].plot(df_CS[mask]['hid_ch'], (df_CS[mask]['valstiff/loss']), '--o')

    mask = np.logical_and(df_CS['stt_ch']==stt_ch, df_CS['n_layer']==5)
    axes[0,2].plot(df_CS[mask]['hid_ch'], (df_CS[mask]['val/loss']), '--o')
    axes[1,2].plot(df_CS[mask]['hid_ch'], (df_CS[mask]['valsmooth/loss']), '--o')
    axes[2,2].plot(df_CS[mask]['hid_ch'], (df_CS[mask]['valstiff/loss']), '--o')

for i, ax in enumerate(axes.flatten()):
    if i<3:
        ax.set_ylim([0.05,0.09])
    else:
        ax.set_ylim([0.05,0.1])

    ax.set_yscale('log')
    ax.xaxis.set_ticks(hid_chs)
    ax.yaxis.set_tick_params(pad=20)
    ax.set_xlabel('hidden features')
    ax.set_ylabel('mean absolute error')


htxt_kwargs = {'rotation': 'horizontal', 'ha': 'center', 'weight': 'bold', 'fontsize': 12}
fig.text(0.21, 0.99, r'$\mathbf{d=1}$', **htxt_kwargs)
fig.text(0.54, 0.99, r'$\mathbf{d=3}$', **htxt_kwargs)
fig.text(0.87, 0.99, r'$\mathbf{d=5}$', **htxt_kwargs)

vtxt_kwargs = {'rotation': 'vertical', 'va': 'center', 'weight': 'bold', 'fontsize': 12}
fig.text(-0.01, 0.85, 'val', **vtxt_kwargs)
fig.text(-0.01, 0.525, 'valsmooth', **vtxt_kwargs)
fig.text(-0.01, 0.2, 'valstiff', **vtxt_kwargs)
fig.legend(stt_chs, title='state variables', loc='lower center', bbox_to_anchor=(0.5,-0.05), ncol=len(stt_chs), alignment='center', frameon=True)
plt.show()

### fig4_complex_stress

In [ ]:
trainer = get_trainer(run_id='olrdx1ao', 
                      run_name='cat8k-long', 
                      data_path='',
                      specify_data=[''],
                      batch_size=2500)

In [ ]:
# find smooth best & worst 
with torch.no_grad():
    most_smooth = None
    trainer.model.eval()

    trainer.batch_size = 2500
    trainer.data_path = ''
    trainer.specify_data = ['smooth']
    trainer.set_dataset(MemoryDataset_LSTM)

    next(trainer.epoch_updater())
    for step in trainer.data_updater('val'):
        y_pred, J_pred, _ = trainer.model.forward_J(trainer.data, trainer.scaler)
        most_smooth = find_most(y_pred, trainer.data.CSdev5, prev=most_smooth, return_index=True)

    err_avg_smooth = (y_pred-trainer.data.CSdev5).abs().mean(dim=(0,-1))
    err_std_smooth = (y_pred-trainer.data.CSdev5).abs().std(dim=(0,-1))

most_smooth[0][0], most_smooth[1][0]

In [ ]:
# find stiff best & worst 
with torch.no_grad():
    most_stiff = None
    trainer.model.eval()

    trainer.batch_size = 2500
    trainer.specify_data = ['stiff']
    trainer.set_dataset(MemoryDataset_LSTM)

    next(trainer.epoch_updater())
    for step in trainer.data_updater('val'):
        y_pred, J_pred, _ = trainer.model.forward_J(trainer.data, trainer.scaler)
        most_stiff = find_most(y_pred, trainer.data.CSdev5, prev=most_stiff)

    # get mae with std
    err_avg_stiff = (y_pred-trainer.data.CSdev5).abs().mean(dim=(0,-1))
    err_std_stiff = (y_pred-trainer.data.CSdev5).abs().std(dim=(0,-1))

most_stiff[0][0], most_stiff[1][0]

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(10, 15), tight_layout=True)

for i, ax in enumerate(axes):
    if i < 2:
        sur_x, sur_y = get_homogenized_xy(most_smooth[i][1].unsqueeze(0))
        sim_x, sim_y = get_homogenized_xy(most_smooth[i][2].unsqueeze(0))
        for j in range(sur_y.shape[-1]):
            line = ax.plot(sur_x[:-9], sur_y[9:,j], '-', alpha=0.8)
            ax.plot(sim_x[:-9], sim_y[9:,j], '--', color=line[0].get_color(), alpha=0.8)
    elif i < 4:
        sur_x, sur_y = get_homogenized_xy(most_stiff[i%2][1].unsqueeze(0))
        sim_x, sim_y = get_homogenized_xy(most_stiff[i%2][2].unsqueeze(0))
        for j in range(sur_y.shape[-1]):
            line = ax.plot(sur_x[:-9], sur_y[9:,j], '-', alpha=0.8)
            ax.plot(sim_x[:-9], sim_y[9:,j], '--', color=line[0].get_color(), alpha=0.8)
    else:
        avg_x, avg_y = get_homogenized_xy(err_avg_smooth.unsqueeze(0).unsqueeze(-1), fill_first=True)
        std_x, std_y = get_homogenized_xy(err_std_smooth.unsqueeze(0).unsqueeze(-1), fill_first=True)
        ax.plot(avg_x[:-9], avg_y[9:], '-', alpha=0.8, label='dataset $S$')
        ax.fill_between(std_x.flatten()[:-9], (avg_y-std_y).flatten()[9:], (avg_y+std_y).flatten()[9:], alpha=0.1)

        avg_x, avg_y = get_homogenized_xy(err_avg_stiff.unsqueeze(0).unsqueeze(-1), fill_first=True)
        std_x, std_y = get_homogenized_xy(err_std_stiff.unsqueeze(0).unsqueeze(-1), fill_first=True)
        ax.plot(avg_x[:-10], avg_y[10:], '-', alpha=0.8, label='dataset $R$')
        ax.fill_between(std_x.flatten()[:-9], (avg_y-std_y).flatten()[9:], (avg_y+std_y).flatten()[9:], alpha=0.1)

        ax.set_xlabel('time[s]')
        ax.set_ylabel('mean absolute error[-]')
        ax.legend(loc='upper left')

    ax.set_xticks(np.linspace(0, 1000, 11), labels=np.linspace(0, 100, 11, dtype=np.int))
    ax.grid(alpha=0.2)

    if i < 4:
        if i==0 or i==2:
            ax.legend(['approx.', 'truth'], loc='lower left')
        else:
            ax.legend(['approx.', 'truth'], loc='upper left')
        ax.get_legend().legendHandles[0].set_color('k')
        ax.get_legend().legendHandles[1].set_color('k')
        ax.set_xlabel('time[s]')
        ax.set_ylabel('normalized outputs[-]')

axes[0].set_title('best performance on dataset $S$', fontweight='bold')
axes[1].set_title('worst performance on dataset $S$', fontweight='bold')
axes[2].set_title('best performance on dataset $R$', fontweight='bold')
axes[3].set_title('worst performance on dataset $R$', fontweight='bold')
axes[4].set_title('average error through time', fontweight='bold')

plt.show()

### fig4_simple_stress

#### main component

In [ ]:
trainer.model.eval()
trainer.batch_size = 2500
trainer.data_path = ''
trainer.specify_data = ['']
trainer.set_dataset(MemoryDataset_LSTM, train_split=0)

next(trainer.epoch_updater())
for step in trainer.data_updater('val'):
    y_pred, J_pred, _ = trainer.model.forward_J(trainer.data, trainer.scaler)

y_pred = trainer.scaler.inverse_transform(y_pred, 'CSdev5')
y = trainer.scaler.inverse_transform(trainer.data.CSdev5, 'CSdev5')
y_pred = dev5_to_m6(y_pred, 0)
y = dev5_to_m6(y, 0)

trainer.data.desc

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), tight_layout=True)

sim_shr_x, sim_shr_y = get_homogenized_xy(y[0].unsqueeze(0).detach(), fill_first=True)
sur_shr_x, sur_shr_y = get_homogenized_xy(y_pred[0].unsqueeze(0).detach(), fill_first=True)
sim_ten_x, sim_ten_y = get_homogenized_xy(y[1].unsqueeze(0).detach(), fill_first=True)
sur_ten_x, sur_ten_y = get_homogenized_xy(y_pred[1].unsqueeze(0).detach(), fill_first=True)
shr_HS, ten_HS = torch.cat([torch.zeros(2,1,6), torch.cumsum(trainer.data.dHS, dim=1)], dim=1)

axes[0].plot(ten_HS[...,0], sim_ten_y[...,0]/1e6, '--', color='darkblue', alpha=0.8)
axes[0].plot(ten_HS[...,0], sur_ten_y[...,0]/1e6, color='darkblue', alpha=0.8)
axes[0].set_xlabel('$\\varepsilon_{11}$')
axes[0].set_ylabel('$\sigma_{11}$[MPa]')

axes[1].plot(shr_HS[...,3], sim_shr_y[...,3]/1e6, '--', color='darkred', alpha=0.8)
axes[1].plot(shr_HS[...,3], sur_shr_y[...,3]/1e6, color='darkred', alpha=0.8)
axes[1].set_xlabel('$\\varepsilon_{12}$')
axes[1].set_ylabel('$\sigma_{12}$[MPa]')

for ax in axes:
    ax.legend(['truth', 'approx.'])
    ax.grid(alpha=0.2)
    
plt.show()

#### full components

In [ ]:
trainer.model.eval()
trainer.batch_size = 2500
trainer.data_path = ''
trainer.specify_data = ['shearxy']
trainer.set_dataset(MemoryDataset_LSTM, train_split=0)

next(trainer.epoch_updater())
for step in trainer.data_updater('val'):
    y_pred, J_pred, _ = trainer.model.forward_J(trainer.data, trainer.scaler)

y_pred = trainer.scaler.inverse_transform(y_pred, 'CSdev5')
y = trainer.scaler.inverse_transform(trainer.data.CSdev5, 'CSdev5')
y_pred = dev5_to_m6(y_pred, trainer.data.CShyd6)/1e6
y = dev5_to_m6(y, trainer.data.CShyd6)/1e6

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(8, 8), tight_layout=True)

for i, ax in enumerate(axes.flatten(order='F')):
    sim_x, sim_y = get_homogenized_xy(y[...,i:i+1].detach())
    sur_x, sur_y = get_homogenized_xy(y_pred[...,i:i+1].detach())
    ax.plot(sim_x[:-10], sim_y[:-10], '--', color='darkred', alpha=0.8)
    ax.plot(sur_x[:-10], sur_y[:-10], color='darkred', alpha=0.8)
    ax.set_xlabel('time[s]')
    ax.set_xticks(np.linspace(0,1000,6), labels=np.linspace(0,100,6, dtype=int))
    ax.set_ylabel(np.array(VIZ.names_IMP6).flatten(order='F')[i])
    ax.legend(['truth', 'approx.'])
    ax.grid(alpha=0.2)

plt.show()

### fig_4_complex_J

In [ ]:
trainer.model.eval()
trainer.batch_size = 2500
trainer.data_path = ''
trainer.specify_data = ['morient_20_16_smooth_501']
trainer.set_dataset(MemoryDataset_LSTM, train_split=0)

next(trainer.epoch_updater())
for step in trainer.data_updater('val'):
    y_pred, J_pred, _ = trainer.model.forward_J(trainer.data, trainer.scaler)

In [ ]:
J_pred = J_pred.reshape(*J_pred.size()[:-1], 6, 6)
J_pred = (J_pred + torch.transpose(J_pred, -2, -1))*0.5
J_pred = J_pred.flatten(-2, -1)

sim_x, sim_y = get_homogenized_xy(trainer.data.dCSdE[:,1:,:].detach(), fill_first=False)
sur_x, sur_y = get_homogenized_xy(J_pred.detach(), fill_first=False)

fig, axes = plt.subplots(6, 6, figsize=(10,9), tight_layout=True)
for i, axs in enumerate(axes):
    for j, ax in enumerate(axs):
        if i >= j:
            ax.plot(sur_x, sur_y[...,i*6+j], '-', alpha=0.8)
            ax.plot(sim_x, sim_y[...,i*6+j], '--', alpha=0.8)
            ax.spines[['right', 'top']].set_visible(False)
        else:
            ax.axis('off')
            
fig.legend(['approx.', 'truth'], 
        #    title='state variables', 
           loc=[0.75, 0.75], 
        #    bbox_to_anchor=(0.5,-0.05), 
        #    ncol=len(stt_chs), 
           alignment='center', 
           frameon=True)
plt.show()

### fig_4_simple_J

In [ ]:
trainer.model.eval()
trainer.batch_size = 2500
trainer.data_path = ''
trainer.specify_data = ['tensionx']
trainer.set_dataset(MemoryDataset_LSTM, train_split=0)

next(trainer.epoch_updater())
for step in trainer.data_updater('val'):
    y_pred, J_pred, _ = trainer.model.forward_J(trainer.data, trainer.scaler)

In [ ]:
J_pred = J_pred.reshape(*J_pred.size()[:-1], 6, 6)
J_pred = (J_pred + torch.transpose(J_pred, -2, -1))*0.5
J_pred = J_pred.flatten(-2, -1)

sim_x, sim_y = get_homogenized_xy(trainer.data.dCSdE[:,1:,:].detach(), fill_first=False)
sur_x, sur_y = get_homogenized_xy(J_pred.detach(), fill_first=False)

fig, axes = plt.subplots(6, 6, figsize=(10,9), tight_layout=True)
for i, axs in enumerate(axes):
    for j, ax in enumerate(axs):
        if i >= j:
            ax.plot(sur_x, sur_y[...,i*6+j], '-', alpha=0.8)
            ax.plot(sim_x, sim_y[...,i*6+j], '--', alpha=0.8)
            ax.spines[['right', 'top']].set_visible(False)
        else:
            ax.axis('off')
            
fig.legend(['approx.', 'truth'], 
        #    title='state variables', 
           loc=[0.75, 0.75], 
        #    bbox_to_anchor=(0.5,-0.05), 
        #    ncol=len(stt_chs), 
           alignment='center', 
           frameon=True)
plt.show()

### fig4_abaqusNN_verify

In [ ]:
def get_abqNN(root, name_base, components=['11','22','33','12','13','23'], logarithmic=False):
    strain, stress = [], []
    for comp in components:
        file = f'{name_base}{comp}.txt'
        df = pd.read_csv(os.path.join(root, file), sep='\s+')
        strain.append(df.iloc[:,0][2:].to_numpy())  # exclude front 2
        stress.append(df.iloc[:,1][2:].to_numpy())  # exclude front 2
    strain = torch.from_numpy(np.array(strain))     # (6, s)
    stress = torch.from_numpy(np.array(stress))     # (6, s)
    strain = strain.t().unsqueeze(0)                # (1, s, 6)
    stress = stress.t().unsqueeze(0)                # (1, s ,6)

    if logarithmic:
        sim_strain = strain  
        sim_stress = stress 
    else:
        sim_strain = np.exp(strain)-1
        sim_stress = stress/(1+sim_strain)
    return sim_strain, sim_stress

#### SEcomplex

In [ ]:
root = r''
# root = r''
# root = r''
name_base = 'SEcomplex_LE-S'
strain, stress = get_abqNN(root, name_base)

data_path = ''
specify_data = ['1554']
y_pred = predict(data_path=data_path, specify_data=specify_data)

# plot_homogenized(stress, y_pred, VIZ.names_IMP6, order='F', allInOne=True, twinx=3, legend=False, figsize=(6,3))
plot_homogenized(stress, y_pred, VIZ.names_IMP6, order='F', legend=False)
plt.show()

#### SEtensionx

In [ ]:
root = r''
name_base = 'SEtensionx_LE-S'
strain, stress = get_abqNN(root, name_base)

data_path = ''
specify_data = ['tensionx']
y_pred = predict(data_path=data_path, specify_data=specify_data)

plot_homogenized(stress[...,0:1], y_pred[...,0:1], VIZ.names_IMP6, order='F', allInOne=True, legend=False, figsize=(6,3))
plt.show()

#### SEtensionxLoad

##### explicit

In [ ]:
root = r''
name_base = 'SEtensionxLoad_LE-S'
strain, stress = get_abqNN(root, name_base)
plot_homogenized(stress, stress, VIZ.names_IMP6, order='F', legend=False)
plt.show()

##### implicit

In [ ]:
root = r''
name_base = 'SEtensionxLoad_LE-S'
strain, stress = get_abqNN(root, name_base)
plot_homogenized(stress, stress, VIZ.names_IMP6, order='F', legend=False)
plt.show()

#### SEshearxy

In [ ]:
root = r''
name_base = 'SEshearxy_LE-S'
strain, stress = get_abqNN(root, name_base)

data_path = ''
specify_data = ['shearxy']
y_pred = predict(data_path=data_path, specify_data=specify_data)

plot_homogenized(stress[...,3:4], y_pred[...,3:4], VIZ.names_IMP6, order='F', allInOne=True, legend=False, figsize=(6,3))
plt.show()

#### CBneural

In [ ]:
root = r''
name_base = 'CBneural_LE-S'
strain, stress = get_abqNN(root, name_base)
dHS = strain[:,1:,:] - strain[:,:-1,:]

K = 7.4972e+10 / 1e6
eng = torch.tensor([1,1,1,2,2,2])
norm = torch.tensor([1,1,1,0,0,0])

# deviatoric
dHSdev5 = dev6_to_dev5(dHS.div(eng))
y_pred = predict(dHSdev5=dHSdev5.float(), CShyd6=0)

# hydrostatic
dHSvol = dHS[...,:3]
dCShyd6 = K*dHSvol.sum(dim=-1, keepdim=True)*norm
CShyd6 = torch.stack([dCShyd6[:,:s+1,:].sum(1) for s in range(dCShyd6.size(1))], dim=1)
CS = y_pred + CShyd6

# plot_homogenized(stress, y_pred, VIZ.names_IMP6, order='F', allInOne=True, twinx=3, legend=False, figsize=(6,3))
plot_homogenized(stress, CS, VIZ.names_IMP6, order='F', legend=False)
plt.show()

#### DBelastic

In [ ]:
root = r''
name_base = 'DBelastic_LE-S'
strain, stress = get_abqNN(root, name_base)
# plot_homogenized(strain, strain, VIZ.names_IMP6, order='F', legend=False)
plot_homogenized(stress, stress, VIZ.names_IMP6, order='F', legend=False)
# plot_homogenized(
#     math_sym33tom6(math_m6toHydDev(stress)[1]), 
#     math_sym33tom6(math_m6toHydDev(stress)[1]), 
#     VIZ.names_IMP6, order='F', legend=False)
plt.show()

#### DBneural

In [ ]:
from intersect import intersection

def getEXP(plastic=False):
    """
    Return
    --------------------
    exp_strain: engineering strain
    exp_stress: engineering stress
    exp_YSx: yield point x 
    exp_YSy: yield point y 
    """
    data = pd.read_excel(f'../../data/exp/AA6111/SScurve/AA6111_new.xlsx')
    idx = data.columns.get_loc(f'7min_298K')
    exp_strain = data.iloc[:,idx+1].dropna().values/100
    exp_stress = data.iloc[:,idx+2].dropna().values
    ylim = max(exp_stress)
    E = 69000 	# MPa, TODO: get E from actual slope
    # E = np.mean(
    # 	[stress/strain for strain, stress in zip(exp_strain,exp_stress) if strain<=0.002 and strain!=0]
    # 	)
    if data.iloc[0,idx]=='correction':
        """
        for incorrect Youngs Modules
        """
        fake_E = exp_stress[1]/exp_strain[1] 	# MPa
        fake_YSx, exp_YSy = intersection(exp_strain, exp_stress,
                                            np.array([0.002, ylim/fake_E+0.002]),
                                            np.array([0,ylim]))
        exp_YSx = exp_YSy/E+0.002
        exp_strain = exp_strain[exp_stress>=exp_YSy]
        exp_stress = exp_stress[exp_stress>=exp_YSy]
        exp_strain -= exp_strain[0]-exp_YSx
    elif data.iloc[0,idx]=='plastic':
        """
        for only plastic strain
        """
        exp_YSx = [exp_strain[0]]
        exp_YSy = [exp_stress[0]]
    else:
        exp_YSx, exp_YSy = intersection(exp_strain, exp_stress,
                                        np.array([0.002, ylim/E+0.002]),
                                        np.array([0,ylim]))

    if plastic:
        plastic_mask = exp_strain>=exp_YSx
        exp_stress = exp_stress[plastic_mask]
        exp_strain = exp_strain[plastic_mask]

    return exp_strain, exp_stress, exp_YSx, exp_YSy

In [ ]:
root = r''
folder = 'element1414'
name_base = 'DBneural_LE-S'
sim_strain, sim_stress = get_abqNN(f'{root}/{folder}', name_base)
# exp_strain, exp_stress, exp_YSx, exp_YSy = getEXP()

root = r''
name_base = 'DBneural_LE-S'
sim_strain_long, sim_stress_long = get_abqNN(f'{root}', name_base)

plt.figure(figsize=(5,3))
plt.plot(sim_strain[0,:,1], sim_stress[0,:,1], label='NAJ model', color='darkblue', alpha=0.8)
plt.plot(sim_strain_long[0,:,1], sim_stress_long[0,:,1], label='J model', color='darkred', alpha=0.8, linestyle='--')
plt.plot(sim_strain_long[0,-1,1], sim_stress_long[0,-1,1], 'x', color='darkred', alpha=0.8)
# plt.plot(exp_strain, exp_stress, label='exp', color='darkred', alpha=0.8)
plt.xlabel('Engineering strain $\\varepsilon_{22}$')
plt.ylabel('Engineering stress $\\sigma_{22}$[MPa]')
plt.legend(loc='lower right')
plt.grid(alpha=0.2)
plt.show()

In [ ]:
root = r''

n3403_n3369_L0 = 50     # (mm)
n3403_disp = pd.read_csv(f'{root}/node3403/DBneural_U2.txt', sep='\s+', header=0)
n3369_disp = pd.read_csv(f'{root}/node3369/DBneural_U2.txt', sep='\s+', header=0)
sim_strain = (n3369_disp['U2'] - n3403_disp['U2'])/n3403_n3369_L0
_, sim_stress = get_abqNN(f'{root}/element1414', 'DBneural_LE-S')

plt.figure(figsize=(5,3))
plt.plot(sim_strain[6:], sim_stress[0,:,1], label='Abaqus+Pytorch', color='darkblue', alpha=0.8)
plt.xlabel('Engineering strain $\\varepsilon_{22}$')
plt.ylabel('Engineering stress $\\sigma_{22}$[MPa]')
plt.legend(loc='lower right')
plt.grid(alpha=0.2)
plt.show()

In [ ]:
path = r''
df = pd.read_csv(f'{path}/DBneural.sta', sep='\s+', skiprows=5, header=None)
t = df.iloc[:-1,6]
itr = df.iloc[:-1,5].astype(int)
cut_idx = [i for i, x in enumerate(df.iloc[:-1,2]) if 'U' in x]

path = r''
df = pd.read_csv(f'{path}/DBneural.sta', sep='\s+', skiprows=5, header=None)
t_long = df.iloc[:-1,6]
itr_long = df.iloc[:-1,5].astype(int)
cut_idx_long = [i for i, x in enumerate(df.iloc[:-1,2]) if 'U' in x]

plt.figure(figsize=(10,3))
plt.plot(t, itr, alpha=0.8, color='darkblue', label='NAJ model')
plt.plot(t.iloc[cut_idx], itr.iloc[cut_idx], '^', c='red')
plt.plot(t_long, itr_long, alpha=0.8, color='darkred', linestyle='--', label='J model')
plt.plot(t_long.iloc[cut_idx_long], itr_long.iloc[cut_idx_long], '^', c='red', label='cutback')
plt.ylim([0, 12])
plt.xticks(np.linspace(0, 1, 11))
plt.yticks(np.linspace(0, 12, 13))

plt.xlabel('fraction')
plt.ylabel('Newton iteraion')
plt.grid(alpha=0.2)
plt.legend()
plt.show()

In [ ]:
path = r''
runs = ['none', 'Jlong', 'noiseAJ']
labels = ['Base model', 'J model', 'NAJ model']
colors = ['r', 'b', 'g']

plt.figure(figsize=(10,3))
for run, label, color in zip(runs, labels, colors):
    df = pd.read_csv(f'{path}_{run}/CBneural.sta', sep='\s+', skiprows=5, header=None)
    t = df.iloc[:-1, 6].values
    dt = df.iloc[:-1, 8].values
    plt.plot(t, dt, '-o', alpha=0.8, label=label, markersize=3, color=color, linewidth=1)
    if t[-1] != 1.0:
        plt.plot(t[-1], dt[-1], 'x', color=color)

plt.xlabel('time[s]')
plt.ylabel('time increment[s]')
plt.yscale('log')
plt.xticks(np.linspace(0, 1, 11))
plt.grid(alpha=0.2, which='both')
plt.legend()
plt.savefig(r'')
plt.show()

In [ ]:
path = r''
runs = ['none', 'Jlong', 'noiseAJ']
labels = ['Base model', 'J model', 'NAJ model']
colors = ['r', 'b', 'g']

plt.figure(figsize=(10,3))
for run, label, color in zip(runs, labels, colors):
    df = pd.read_csv(f'{path}_{run}/DBneural.sta', sep='\s+', skiprows=5, header=None)
    t = df.iloc[:-1, 6].values
    dt = df.iloc[:-1, 8].values
    plt.plot(t, dt, '-o', alpha=0.8, label=label, markersize=3, color=color, linewidth=1)
    if t[-1] != 1.0:
        plt.plot(t[-1], dt[-1], 'x', color=color)

plt.xlabel('time[s]')
plt.ylabel('time increment[s]')
plt.yscale('log')
plt.xticks(np.linspace(0, 1, 11))
plt.yticks()
plt.grid(alpha=0.2, which='both')
plt.legend()
plt.savefig(r'')
plt.show()

### fig4_strainInc_verify

In [ ]:
MULT = 1000
F_intp = interpolate(F.mean(dim=1).permute(1,2,0), size=101*MULT, mode='linear').permute(2,0,1)
CS_intp = interpolate(CS.mean(dim=1).permute(1,2,0), size=101*MULT, mode='linear').permute(2,0,1)
result_intp = FCS_to_dev5(F_intp, CS_intp, dim=0)
result_intp.keys()

In [ ]:
# inc_101
data_path = ''
specify_data = ['1554']
y_pred = predict(data_path=data_path, specify_data=specify_data, exp_name='LMSC_test', run_id='c91291dc00d746c7932e07ef2122e6c4')

# inc_101*MULT
dHSdev5_intp = result_intp['dHSdev5'].unsqueeze(0).float()
CShyd6_intp = result_intp['CShyd6'].unsqueeze(0)
y_pred_intp = predict(dHSdev5=dHSdev5_intp, CShyd6=CShyd6_intp, exp_name='LMSC_test', run_id='c91291dc00d746c7932e07ef2122e6c4')

plot_homogenized(y_pred, y_pred_intp[:,MULT-1::MULT], VIZ.names_IMP6, order='F')
plt.show()

In [ ]:
# zero increment 
dHSdev5_zero = torch.zeros((1,100,5), dtype=torch.float32)
y_pred_zero = predict(dHSdev5=dHSdev5_zero)

plot_homogenized(y_pred_zero, y_pred_zero, VIZ.names_IMP6, order='F')
plt.show()

### loading-unloading verify

In [ ]:
# elastic loading-unloading
cycles = 10
dHSdev5_cycles = []
for _ in range(cycles): 
    dHSdev5_cycles.append(torch.zeros((1, 10, 5), dtype=torch.float32) + 1e-8)
    dHSdev5_cycles.append(torch.zeros((1, 10, 5), dtype=torch.float32) - 1e-8)
dHSdev5_cycles = torch.cat(dHSdev5_cycles, dim=1)
y_pred_cycle = predict(dHSdev5=dHSdev5_cycles, CShyd6=0)

plot_homogenized(y_pred_cycle, y_pred_cycle, VIZ.names_IMP6, order='F')
plt.show()

### check LMSC aux

#### alpha

In [ ]:
# # none
# alpha = predict(
#     exp_name='LMSC',
#     run_id='7af51c503f194c8f8dd140138e667fae',
#     data_path='', 
#     save_root='',
#     specify_data=['tensionx'], 
#     backend='mlflow',
#     return_aux=True, CShyd6=0)[2]
# plot_homogenized(alpha, alpha, [['dummy']]*32, legend=False, allInOne=True, figsize=(8,3), fill_first=False)

# concat_noiseJ
pred = predict(
    exp_name='LMSC',
    run_name='NJAinf',
    run_id='wdg32otq',
    data_path='', 
    specify_data=['tensionx'], 
    return_aux=True, CShyd6=0)

y_pred, alpha = pred[0], pred[2]
plot_homogenized(alpha, alpha, [['dummy']]*32, legend=False, allInOne=True, figsize=(8,3), fill_first=False)
plot_homogenized(y_pred, y_pred, VIZ.names_IMP6, legend=False, allInOne=True, figsize=(8,3), fill_first=False)

plt.show()

In [ ]:
fig, ax1 = plt.subplots(1, figsize=(15,3))
img = ax1.imshow(alpha.detach()[0].t(), cmap='Reds', norm='linear')
ax1.set_xlabel('increment')
ax1.set_ylabel('state variable')

# ax2 = ax1.twinx()
# y_pred = y_pred.detach().cpu()
# ax2.plot(torch.arange(y_pred.size(1)), y_pred[0,:,0]/1e6,'--o', color='skyblue')
# ax2.set_ylabel('$\sigma_{11}[MPa]$')

fig.colorbar(img, ax=ax1)
plt.show()

#### beta

In [ ]:
# none
beta = predict(
    data_path='', 
    specify_data=['tensionx'], 
    exp_name='LMSC_tune',
    run_id='eae4eae67c10469d890b14781c69e6de',
    return_aux=True, CShyd6=0)[3]
plot_homogenized(beta, beta, [['dummy']]*32, legend=False, allInOne=True, figsize=(15,5), fill_first=False)

# noiseAJ
beta = predict(
    data_path='', 
    specify_data=['tensionx'], 
    exp_name='LMSC',
    run_id='1304699d15ac4869acc16d27311af658',
    return_aux=True, CShyd6=0)[3]
plot_homogenized(beta, beta, [['dummy']]*32, legend=False, allInOne=True, figsize=(15,5), fill_first=False)

plt.show()

### elastic stiffness verify

In [ ]:
trainer = get_trainer()

#### vmap + jacrev

In [ ]:
from torch.autograd.functional import jacobian
from functorch import vmap, jacrev  
torch.set_printoptions(precision=2)

def forward_J(dHS):
    K = 7.4972e+10
    
    dHS_ = dHS.div(torch.tensor([1,1,1,2,2,2]))
    
    # deviatoric
    dHSdev5 = dev6_to_dev5(dHS_)
    CSdev6 = predict(dHSdev5=dHSdev5, CShyd6=0, grad=True)*1e6
    dCSdev6 = CSdev6
    dCSdev6[:,1:,:] = dCSdev6[:,1:,:] - dCSdev6[:,:-1,:]

    # hydrostatic
    dHSvol = dHS_[...,:3]
    dCShyd6 = K*dHSvol.sum(dim=-1, keepdim=True)*torch.tensor([1,1,1,0,0,0])
    
    # assemble
    dCS = CSdev6 + dCShyd6
    return dCS.sum(0).sum(0)

In [ ]:
# test 
threshold = 1e-6
dHS = torch.tensor([threshold,threshold,threshold,threshold,threshold,threshold])
dHS = dHS.repeat(1,1,1).requires_grad_()
# J = vmap(vmap(jacrev(forward_J)))(dHS)
J = jacobian(forward_J, dHS, vectorize=True).permute(1,2,0,3)
J.size(), J

#### vmap + grad 

In [ ]:
from functorch import vmap 
torch.set_printoptions(precision=2)

def predict_J(dHS, return_graph=False):
    K = 7.4972e+10
    model = predict(return_model=True)
    dHS_ = dHS.requires_grad_().div(torch.tensor([1,1,1,2,2,2]))

    # deviatoric
    dHSdev5 = dev6_to_dev5(dHS_)                             # (b, s, c)
    y_pred = model(dHSdev5)[0]
    y_pred = model.scaler.inverse_transform(y_pred, 'CSdev5')
    CSdev6 = dev5_to_m6(y_pred, 0)
    dCSdev6 = CSdev6                                            # this is dummy (?)
    dCSdev6[:,1:,:] = dCSdev6[:,1:,:] - dCSdev6[:,:-1,:]        # this is dummy (?)

    # hydrostatic
    dHSvol = dHS_[...,:3]
    dCShyd6 = K*dHSvol.sum(dim=-1, keepdim=True)*torch.tensor([1,1,1,0,0,0])
    
    # assemble
    dCS = dCSdev6 + dCShyd6
    J = dCSdHS_to_DDSDDE(dCS, dHS, symmetrise=True, create_graph=return_graph)

    if return_grapF:
        graph = {
            'dHS': dHS, 'dHS_': dHS_, 'dHSdev5': dHSdev5, 'dHSvol': dHSvol,
            'y_pred': y_pred, 'CSdev6': CSdev6, 'dCSdev6': dCSdev6, 'dCShyd6': dCShyd6, 'dCS': dCS, 'J': J
        }
        graph.update(dict(model.named_parameters()))
        return J, graph
    
    return J

In [ ]:
# inp = torch.tensor([7.5258e-03,  4.2692e-03,  4.4359e-03,  2.1193e-03, -3.5609e-03, 4.8206e-03])
inp = torch.tensor([7e-5, 7e-5, 7e-5, 7e-5, 7e-5, 7e-5]).repeat(1,100,1)
J_pred = predict_J(inp)
J_pred.size(), J_pred[0,:,0,0]

#### vmap + grad (recursively)

In [ ]:
from functorch import vmap 
# torch.set_printoptions(precision=2)

def predict_J(dHS_t, return_graph=False, double_precision=True):
    K = 7.4972e+10
    eng = torch.tensor([1,1,1,2,2,2])
    norm = torch.tensor([1,1,1,0,0,0])
    model = trainer.model

    if double_precision:
        model = model.double()
        dHS_t = dHS_t.double()

    J = []
    for s in range(dHS_t.size(1)):
        dHS = dHS_t[:,s:s+1,:].mul(eng).requires_grad_()
        dHS_ = dHS.div(eng)

        # deviatoric
        dHSdev5 = m6_to_dev5(dHS_)                                  # (b, s, c)
        y_pred, chi, _, _ = model(dHSdev5, chi=None if s==0 else chi)
        y_pred = trainer.scaler.inverse_transform(y_pred, 'CSdev5')
        CSdev6 = dev5_to_m6(y_pred, 0)
        dCSdev6 = CSdev6                                            # this is dummy (?)
        dCSdev6[:,1:,:] = dCSdev6[:,1:,:] - dCSdev6[:,:-1,:]        # this is dummy (?)

        # hydrostatic
        dHSvol = dHS_[...,:3]
        dCShyd6 = K*dHSvol.sum(dim=-1, keepdim=True)*norm
        
        # assemble
        dCS = dCSdev6 + dCShyd6
        J.append(dCSdHS_to_DDSDDE(dCS, dHS, symmetrise=True, create_graph=return_graph))
    J = torch.cat(J, dim=1)

    if return_grapF:
        graph = {
            'dHS': dHS, 'dHS_': dHS_, 'dHSdev5': dHSdev5, 'dHSvol': dHSvol,
            'y_pred': y_pred, 'CSdev6': CSdev6, 'dCSdev6': dCSdev6, 'dCShyd6': dCShyd6, 'dCS': dCS, 'J': J
        }
        graph.update(dict(model.named_parameters()))
        return J, graph
    
    return J

#### fig4_J_damping (need to check)

In [ ]:
# tensionx
next(trainer.epoch_updater())
next(trainer.data_updater('test'))
inp0 = trainer.data.dHS[:,0:1,:].double()

# artificial 
# inp0 = torch.tensor([0,0,0,0,0,0]).double().unsqueeze(0).unsqueeze(0)

inp0 = torch.cat([torch.zeros_like(inp0), inp0], dim=1)
inp0_interp = interpolate(inp0.permute(0,2,1), size=1000, mode='linear').permute(0,2,1)     # (b, s, 6)

J_pred = [predict_J(inp0_interp[:,t:t+1,:]).squeeze() for t in range(inp0_interp.size(1))]
J_pred = torch.stack(J_pred, dim=0)                                                         # (s, 6, 6)

In [ ]:
J_pred.size(), J_pred[:,0,0]

In [ ]:
x = torch.linalg.norm(inp0_interp[0], dim=-1, keepdim=True)     # (s, 1)
y = J_pred[:,[0,3,0],[0,3,1]]                                   # (s, 3)
x_crit = x[y[:,1].argmax()]

plt.plot(x, y/1e9, label=['$C_{11}$', '$C_{44}$', '$C_{12}$'])
plt.plot([x_crit, x_crit], [0, 1e3], '--k', alpha=0.5)
plt.fill_betweenx([0, 1e3], 0, x_crit, color='salmon', alpha=0.1)
plt.text(1.1*x_crit, 1, f'critical strain norm \n {x_crit.item():.2e}', ha='left')

plt.yscale('log')
plt.xlim([0, 5e-6])
plt.ylim([1e-1, 1e3])
plt.xlabel('strain norm')
plt.ylabel('stiffness[GPa]')
plt.grid(alpha=0.2)
plt.legend()
plt.show()

In [ ]:
# value should be added to each component of dHS6
x_crit.square().mul(3).div(2).div(6).sqrt()

#### plot gradient graph 

In [ ]:
inp = torch.tensor([7e-5, 7e-5, 7e-5, 7e-5, 7e-5, 7e-5])
dCS, graph = predict_J(inp, return_graph=True)
torchviz.make_dot(dCS, graph, show_attrs=True)

In [ ]:
model = predict(return_model=True)

dHS = torch.tensor([7e-5, 7e-5, 7e-5, 7e-5, 7e-5, 7e-5])
dHS = dHS.repeat(1,2,1).div(torch.tensor([1,1,1,2,2,2]))
dHSdev5 = dev6_to_dev5(dHS)
y_pred = model(dHSdev5.requires_grad_())[0]

graph = {'CSdev5': y_pred, 'dHSdev5': dHSdev5}
graph.update(dict(model.named_parameters()))
torchviz.make_dot(y_pred, graph, show_attrs=True)

### computational efficiency

In [ ]:
# element numbers vs. wall time (s)

# abqNN/implicit bulk tensile 
abqBK_ip = [e**3 for e in [1,2,4,8,16,32]]
abqBK_ts = [5,11,99,529,7955,48177]
plt.plot(abqBK_ip, abqBK_ts, '-o', label='standard bulk')

# abqNN/implicit dogbone
abqDB_ip = [3293]
abqDB_ts = [21031]
plt.plot(abqDB_ip, abqDB_ts, '^', label='standard dogbone')

# abqDK_ip = [e**3 for e in [1,2,4,8,16,32]]
# abqDK_ts = [313*mult for mult in abqDK_ip]
# plt.plot(abqDK_ip, abqDK_ts, '--o', label='Abaqus + Damask(estimate)')

plt.xscale('log') 
plt.yscale('log') 
plt.xlabel('macro integraion point number')
plt.ylabel('wall time(s)')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

In [ ]:
trainer = get_trainer(run_id='olrdx1ao', 
                      run_name='cat8k-long', 
                      data_path='',
                      specify_data=[''],
                      batch_size=1,
                      device='cuda')

In [ ]:
import time 

trainer.batch_size = 1
trainer.set_dataset(MemoryDataset_LSTM)
trainer.model.eval()
next(trainer.epoch_updater())
next(trainer.data_updater('val'))
trainer.data

In [ ]:
# cpu
trainer.model.cpu()

s = time.time()

# with torch.no_grad():
#     for _ in range(1):
#         trainer.model.forward(trainer.data.dHSdev5.cpu())

for _ in range(1):
    trainer.model.forward_J(trainer.data.to('cpu'), trainer.scaler.to('cpu'))

f = time.time()
print((f-s)/1)

In [ ]:
# cuda
trainer.model.cuda()

s = time.time()

# with torch.no_grad():
#     for _ in range(1):
#         trainer.model.forward(trainer.data.dHSdev5.cuda())

for _ in range(1):
    trainer.model.forward_J(trainer.data.to('cuda'), trainer.scaler.to('cuda'))

f = time.time()
print((f-s)/1)

In [ ]:
iters = [3,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,2,2,2,2,2,2,2,2,1,2,1,1,2,1,1,1,1,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,3,3,5,5,5,5,4,4,4,4,4,4,3,3,4,3,3,3,3,3,3,3,3,3,3,2,3,2,3,2,3,2,3,2,3,2,3,2,2,3,2,2,3,2,2,3,2,2,3,2,2,3,2,2,3,2,2,3,2,2,2]
sum(iters)

In [ ]:
# damask time cost per iter 
time = 230
time/sum(iters)

In [ ]:
# DBneural
itr.sum()*3293*0.8518518518518519

In [ ]:
# CBneural
itr.sum()*2160*0.8518518518518519